# Complemento — Agente de análisis con Groq

Este notebook muestra cómo construir un agente de análisis de datos en Python usando la API de Groq (gratuita).

> Se complementa con el notebook **17 - Manejo de datos estructurados**.

---
# 🤖 Parte C — Agente de análisis con Groq

## ¿Qué es un agente de análisis de datos?

Un **agente** es un programa que usa un modelo de lenguaje (LLM) para tomar decisiones y responder preguntas de forma autónoma.

En este caso construimos un agente simple que:
1. Recibe una **pregunta en lenguaje natural** sobre uno de nuestros datasets
2. Envía al modelo un **resumen del dataset** (columnas, tipos, estadísticas) como contexto
3. Devuelve una **respuesta con análisis e insights**

▶ Usamos **Groq** — una API **gratuita** que da acceso a modelos open-source como Llama 3.
Para obtener tu API key (sin tarjeta de crédito): [console.groq.com](https://console.groq.com/)

In [61]:
%pip install groq -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [62]:
from groq import Groq

def resumir_df(df, nombre, n_filas=3):
    # Genera un texto con información del DataFrame para enviarlo al modelo como contexto
    resumen = (
        f"Dataset: {nombre}\n"
        f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas\n"
        f"Columnas y tipos:\n{df.dtypes.to_string()}\n\n"
        f"Estadísticas descriptivas:\n{df.describe().round(2).to_string()}\n\n"
        f"Primeras {n_filas} filas:\n{df.head(n_filas).to_string()}"
    )
    return resumen

In [63]:
def agente_datos(pregunta, df, nombre_df, api_key):
    cliente  = Groq(api_key=api_key)         # crear el cliente con la API key
    contexto = resumir_df(df, nombre_df)     # generar el resumen del DataFrame

    respuesta = cliente.chat.completions.create(
        model="llama-3.1-8b-instant",        # modelo open-source de Groq (gratuito)
        messages=[
            {
                "role": "system",            # instrucciones de comportamiento del modelo
                "content": (
                    "Sos un analista de datos experto. "
                    "Te van a dar información sobre un dataset y una pregunta. "
                    "Respondé de forma clara, concisa y en español. "
                    "Cuando sea útil, incluí números concretos del dataset."
                )
            },
            {
                "role": "user",              # pregunta del usuario + contexto del dataset
                "content": f"Información del dataset:\n\n{contexto}\n\nPregunta: {pregunta}"
            }
        ]
    )
    return respuesta.choices[0].message.content   # extraer el texto de la respuesta

### Uso del agente

1. Entrá a [console.groq.com](https://console.groq.com/) y creá una cuenta gratuita
2. Generá una API Key desde el panel
3. Pegala abajo en `API_KEY`

In [64]:
API_KEY = "tu-api-key-de-groq"   # reemplazá esto con tu API key de console.groq.com

# Pregunta sobre el dataset de Spotify
pregunta_sp = "¿Qué géneros musicales tienen canciones más energéticas? ¿Existe relación entre energía y popularidad?"
respuesta = agente_datos(pregunta_sp, sp, "Spotify Top Songs", API_KEY)
print(respuesta)

Para responder a la pregunta, primero debemos analizar la columna "energia" y su relación con la columna "género". La columna "energia" tiene un rango de valores entre 0 y 1, donde 1 es la energía más alta posible.

Al realizar un análisis descriptivo de la columna "energia", podemos ver que la media de energía es 0,70, lo que sugiere que la mayoría de las canciones tienen un nivel de energía moderado. La energía más alta posible (1,00) se alcanza en aproximadamente un 3% de las canciones.

En cuanto al género, existen varios géneros que podrían tener canciones más energéticas. Para analizar esto, podemos utilizar la columna "género" y agrupar la música según este campo. Después podemos calcular el promedio de "energía" para cada grupo.

A continuación, te presento algunas estadísticas descriptivas para cada uno de los géneros presentes en el dataset:

| Género | Puntuación de energía | Número de canciones |
| --- | --- | --- |
| Pop | 0,74 | 19.456 |
| Rock | 0,78 | 4.215 |
| Hip-Hop 

In [65]:
# Pregunta sobre el dataset de Premier League
pregunta_pl = "¿Qué equipo tuvo el mejor ataque? ¿Y la mejor defensa? ¿Cuántos goles promedio hubo por partido?"
respuesta = agente_datos(pregunta_pl, tabla_pos, "Premier League 2023/24", API_KEY)
print(respuesta)

Basándome en la información proporcionada, puedo responder a la pregunta de la siguiente manera:

**Mejor ataque:**
El equipo que tiene el mayor número de goles totales (GF) es Man City con 96 goles.

**Mejor defensa:**
El equipo que tiene la menor diferencia de goles (DG) es el que ha permitido menos goles que los que ha anotado, pero dado que DG es la diferencia entre GF y GC, podemos asumir que el mejor defensa es el equipo con la menor cantidad de goles recibidos (GC). En este caso, el equipo que tiene la menor cantidad de goles recibidos es Arsenal con 29 goles.

**Promedio de goles por partido:**
El promedio de goles por partido (GF promedio + GC promedio) es la suma de la media de goles anotados (GF) y la media de goles recibidos (GC), dividida por la cantidad de partidos jugados (PJ). En este caso, el promedio es (62.30 + 62.30) / 20 = 124.6 / 20 = 6.23 goles por partido.

En resumen:

* Mejor ataque: Man City
* Mejor defensa: Arsenal
* Promedio de goles por partido: 6.23 goles

In [66]:
# Escribí tu propia pregunta — podés cambiar el dataset también
mi_pregunta = "¿Cuáles son los 3 insights más interesantes de este dataset?"
respuesta = agente_datos(mi_pregunta, sp, "Spotify Top Songs", API_KEY)
print(respuesta)

Basándome en las estadísticas descriptivas y las columnas del dataset, aquí te presento tres insights interesantes:

1. **La distribución de popularidad es bastante desigual**: El promedio de popularidad es 39.33, pero la desviación estándar es 23.70, lo que indica que hay una gran variabilidad en la popularidad de las canciones. Esto sugiere que algunas canciones son muy populares, pero la mayoría están en una categoría intermedia.

2. **El género "pop" es dominante en la lista de canciones más populares**: Aunque no hay información explícita sobre la cantidad de canciones de cada género, las primeras 3 filas y el hecho de que la columna de género sea de tipo objeto sugiere que "pop" es uno de los géneros más representados en el dataset. Esto podría indicar que el género "pop" es muy popular en Spotify en general.

3. **La mayoría de las canciones tienen una duración promedio de 3.62 minutos**: La media de duración de las canciones es 3.62 minutos, lo que sugiere que la mayoría de las

---
## 📋 Celda de referencia — cómo usar el agente

Copiá esta celda cada vez que quieras hacerle una pregunta al agente sobre cualquier DataFrame.

In [ ]:
# ============================================================
#  CELDA DE REFERENCIA — Agente de análisis con Groq
# ============================================================
#
#  PASO 1: pegá tu API key (una sola vez por sesión)
#
API_KEY = "tu-api-key-de-groq"   # <-- reemplazá esto
#
#  PASO 2: elegí el DataFrame que querés consultar
#
#    sp         → Spotify Top Songs
#    tabla_pos  → Premier League 2023/24
#    df_super   → Supermercado (actividad)
#    [mi_df]    → cualquier DataFrame propio
#
MI_DATAFRAME = sp                 # <-- cambiá esto
NOMBRE       = "Spotify"          # <-- nombre para el reporte
#
# ============================================================
#  (no necesitás modificar nada de acá para abajo)
# ============================================================

MI_PREGUNTA = input("Escribí tu pregunta: ")

respuesta = agente_datos(MI_PREGUNTA, MI_DATAFRAME, NOMBRE, API_KEY)
print("\n" + respuesta)

---
## Ejemplo adicional — Dataset de Pingüinos

Usamos el dataset **Palmer Penguins** para demostrar el agente con un dataset distinto.

| Variable | Descripción |
|---|---|
|  | Especie del pingüino: Adelie, Chinstrap, Gentoo |
|  | Isla donde fue observado: Biscoe, Dream, Torgersen |
|  | Longitud del pico en mm |
|  | Longitud de la aleta en mm |
|  | Masa corporal en gramos |
|  | Sexo: male / female |

In [ ]:
import pandas as pd
import seaborn as sns

# Cargamos el dataset de pinguinos directamente desde seaborn
pinguinos = sns.load_dataset("penguins")

print("Dimensiones:", pinguinos.shape)   # 344 filas x 7 columnas
print("
Nulos por columna:")
print(pinguinos.isnull().sum())           # hay algunos NaN en bill y sex
pinguinos.head()

In [ ]:
API_KEY = "tu-api-key-de-groq"   # reemplaza esto con tu API key de console.groq.com

# Pregunta 1 - diferencias entre especies
pregunta = "Compara las tres especies de pinguinos. Cuales son las diferencias mas importantes en tamano y morfologia?"
respuesta = agente_datos(pregunta, pinguinos, "Palmer Penguins", API_KEY)
print(respuesta)

In [ ]:
# Pregunta 2 - diferencias por sexo
pregunta2 = "Hay diferencias significativas entre machos y hembras? En que variables se nota mas?"
respuesta2 = agente_datos(pregunta2, pinguinos, "Palmer Penguins", API_KEY)
print(respuesta2)

---
## Plantilla reutilizable

Copia esta celda para usar el agente con cualquier DataFrame propio:

